# Betting Strategies & Evaluation

**Chapter 9: From Predictions to Profit: Navigating the World of Soccer Betting**

## Learning Objectives

- Understand baseline betting strategies (always home, lowest odds)
- Implement fixed stake betting
- Calculate Return on Investment (ROI)
- Track performance over time
- Backtest strategies on historical data
- Compare different betting approaches
- Understand why accuracy ≠ profitability

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("Betting Strategies & Evaluation Toolkit Ready!")

## Introduction

**Accuracy is NOT the same as profitability!**

You can be right 60% of the time and still lose money. Why?
- Bookmaker's margin (overround)
- Betting on favorites with low odds
- Poor bankroll management

This notebook teaches you how to:
1. Establish baseline strategies
2. Calculate true profitability (ROI)
3. Backtest strategies rigorously
4. Compare different approaches

## Generate Match Data

Let's create a season of matches with realistic odds and outcomes.

In [ ]:
# Generate Premier League-style season (380 matches)
np.random.seed(42)

num_matches = 380
matches = []

for i in range(num_matches):
    # Simulate realistic odds with home advantage
    home_win_prob_true = np.random.uniform(0.35, 0.55)
    draw_prob_true = np.random.uniform(0.20, 0.30)
    away_win_prob_true = 1 - home_win_prob_true - draw_prob_true
    
    # Add bookmaker margin (5%)
    overround = 1.05
    home_odds = (1 / home_win_prob_true) * overround
    draw_odds = (1 / draw_prob_true) * overround
    away_odds = (1 / away_win_prob_true) * overround
    
    # Simulate actual outcome
    outcome_rand = np.random.random()
    if outcome_rand < home_win_prob_true:
        result = 'H'
    elif outcome_rand < home_win_prob_true + draw_prob_true:
        result = 'D'
    else:
        result = 'A'
    
    matches.append({
        'match_id': i + 1,
        'home_team': f'Team {(i % 20) + 1}',
        'away_team': f'Team {((i + 10) % 20) + 1}',
        'home_odds': home_odds,
        'draw_odds': draw_odds,
        'away_odds': away_odds,
        'result': result
    })

matches_df = pd.DataFrame(matches)

print(f"Generated {len(matches_df)} matches")
print(f"\nActual Results Distribution:")
print(matches_df['result'].value_counts())
print(f"\nHome Win %: {(matches_df['result'] == 'H').mean():.1%}")
print(f"Draw %: {(matches_df['result'] == 'D').mean():.1%}")
print(f"Away Win %: {(matches_df['result'] == 'A').mean():.1%}")

matches_df.head(10)

---

# 1. Baseline Strategies

Before building complex models, we need **baselines** to beat.

## 1.1 Strategy 1: Always Bet Home

Exploits home advantage - home teams win ~45-50% of matches.

In [ ]:
def strategy_always_home(matches_df, stake=10):
    """
    Always bet on home team to win.
    """
    results = []
    
    for _, match in matches_df.iterrows():
        prediction = 'H'
        odds = match['home_odds']
        actual = match['result']
        
        # Calculate profit/loss
        if prediction == actual:
            profit = (odds - 1) * stake
        else:
            profit = -stake
        
        results.append({
            'match_id': match['match_id'],
            'prediction': prediction,
            'actual': actual,
            'odds': odds,
            'stake': stake,
            'profit': profit,
            'correct': prediction == actual
        })
    
    return pd.DataFrame(results)

# Run strategy
stake = 10
results_home = strategy_always_home(matches_df, stake)

# Calculate metrics
accuracy = results_home['correct'].mean()
total_staked = results_home['stake'].sum()
total_profit = results_home['profit'].sum()
roi = (total_profit / total_staked) * 100

print("Strategy: Always Bet Home\n")
print(f"Accuracy: {accuracy:.1%}")
print(f"Total Staked: ${total_staked:,.0f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"ROI: {roi:+.2f}%")
print(f"\n{'✅ Profitable!' if total_profit > 0 else '❌ Losing strategy'}")

## 1.2 Strategy 2: Always Bet Lowest Odds

Trust the market - bet on the favorite (lowest odds).

In [ ]:
def strategy_lowest_odds(matches_df, stake=10):
    """
    Always bet on outcome with lowest odds (market favorite).
    """
    results = []
    
    for _, match in matches_df.iterrows():
        # Find lowest odds
        odds_dict = {
            'H': match['home_odds'],
            'D': match['draw_odds'],
            'A': match['away_odds']
        }
        
        prediction = min(odds_dict, key=odds_dict.get)
        odds = odds_dict[prediction]
        actual = match['result']
        
        # Calculate profit/loss
        if prediction == actual:
            profit = (odds - 1) * stake
        else:
            profit = -stake
        
        results.append({
            'match_id': match['match_id'],
            'prediction': prediction,
            'actual': actual,
            'odds': odds,
            'stake': stake,
            'profit': profit,
            'correct': prediction == actual
        })
    
    return pd.DataFrame(results)

# Run strategy
results_lowest = strategy_lowest_odds(matches_df, stake)

# Calculate metrics
accuracy = results_lowest['correct'].mean()
total_staked = results_lowest['stake'].sum()
total_profit = results_lowest['profit'].sum()
roi = (total_profit / total_staked) * 100

print("Strategy: Always Bet Lowest Odds\n")
print(f"Accuracy: {accuracy:.1%}")
print(f"Total Staked: ${total_staked:,.0f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"ROI: {roi:+.2f}%")
print(f"\n{'✅ Profitable!' if total_profit > 0 else '❌ Losing strategy'}")

## 1.3 Strategy 3: Home + Lowest Odds

Combine both: only bet home when it's also the favorite.

In [ ]:
def strategy_home_and_lowest(matches_df, stake=10):
    """
    Bet on home team only when it has lowest odds.
    """
    results = []
    
    for _, match in matches_df.iterrows():
        # Check if home has lowest odds
        if match['home_odds'] < match['draw_odds'] and match['home_odds'] < match['away_odds']:
            prediction = 'H'
            odds = match['home_odds']
            actual = match['result']
            
            # Calculate profit/loss
            if prediction == actual:
                profit = (odds - 1) * stake
            else:
                profit = -stake
            
            results.append({
                'match_id': match['match_id'],
                'prediction': prediction,
                'actual': actual,
                'odds': odds,
                'stake': stake,
                'profit': profit,
                'correct': prediction == actual
            })
    
    return pd.DataFrame(results)

# Run strategy
results_combined = strategy_home_and_lowest(matches_df, stake)

# Calculate metrics
if len(results_combined) > 0:
    accuracy = results_combined['correct'].mean()
    total_staked = results_combined['stake'].sum()
    total_profit = results_combined['profit'].sum()
    roi = (total_profit / total_staked) * 100
    
    print("Strategy: Home + Lowest Odds\n")
    print(f"Matches Bet: {len(results_combined)} / {len(matches_df)}")
    print(f"Accuracy: {accuracy:.1%}")
    print(f"Total Staked: ${total_staked:,.0f}")
    print(f"Total Profit: ${total_profit:,.2f}")
    print(f"ROI: {roi:+.2f}%")
    print(f"\n{'✅ Profitable!' if total_profit > 0 else '❌ Losing strategy'}")
else:
    print("No matches met the criteria!")

## 1.4 Compare All Baseline Strategies

In [ ]:
# Create comparison dataframe
comparison = pd.DataFrame({
    'Strategy': ['Always Home', 'Lowest Odds', 'Home + Lowest'],
    'Matches Bet': [
        len(results_home),
        len(results_lowest),
        len(results_combined)
    ],
    'Accuracy': [
        results_home['correct'].mean(),
        results_lowest['correct'].mean(),
        results_combined['correct'].mean() if len(results_combined) > 0 else 0
    ],
    'Total Profit': [
        results_home['profit'].sum(),
        results_lowest['profit'].sum(),
        results_combined['profit'].sum() if len(results_combined) > 0 else 0
    ],
    'ROI (%)': [
        (results_home['profit'].sum() / results_home['stake'].sum()) * 100,
        (results_lowest['profit'].sum() / results_lowest['stake'].sum()) * 100,
        (results_combined['profit'].sum() / results_combined['stake'].sum()) * 100 if len(results_combined) > 0 else 0
    ]
})

print("Baseline Strategies Comparison:\n")
print(comparison.to_string(index=False))

print("\n⚠️  Key Insight: High accuracy doesn't guarantee profitability!")

---

# 2. Performance Tracking Over Time

Visualize how bankroll evolves throughout the season.

In [ ]:
def calculate_cumulative_profit(results_df):
    """
    Calculate cumulative profit over time.
    """
    results_df = results_df.copy()
    results_df['cumulative_profit'] = results_df['profit'].cumsum()
    results_df['cumulative_roi'] = (results_df['cumulative_profit'] / 
                                     (results_df['stake'].cumsum())) * 100
    return results_df

# Calculate cumulative metrics
results_home_cum = calculate_cumulative_profit(results_home)
results_lowest_cum = calculate_cumulative_profit(results_lowest)
if len(results_combined) > 0:
    results_combined_cum = calculate_cumulative_profit(results_combined)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Cumulative profit
ax1.plot(results_home_cum['match_id'], results_home_cum['cumulative_profit'], 
         linewidth=2, label='Always Home', alpha=0.8)
ax1.plot(results_lowest_cum['match_id'], results_lowest_cum['cumulative_profit'], 
         linewidth=2, label='Lowest Odds', alpha=0.8)
if len(results_combined) > 0:
    ax1.plot(results_combined_cum.index, results_combined_cum['cumulative_profit'], 
             linewidth=2, label='Home + Lowest', alpha=0.8)

ax1.axhline(0, color='black', linestyle='--', alpha=0.5)
ax1.set_xlabel('Match Number', fontsize=12)
ax1.set_ylabel('Cumulative Profit ($)', fontsize=12)
ax1.set_title('Bankroll Evolution', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Right panel: ROI over time
ax2.plot(results_home_cum['match_id'], results_home_cum['cumulative_roi'], 
         linewidth=2, label='Always Home', alpha=0.8)
ax2.plot(results_lowest_cum['match_id'], results_lowest_cum['cumulative_roi'], 
         linewidth=2, label='Lowest Odds', alpha=0.8)
if len(results_combined) > 0:
    ax2.plot(results_combined_cum.index, results_combined_cum['cumulative_roi'], 
             linewidth=2, label='Home + Lowest', alpha=0.8)

ax2.axhline(0, color='black', linestyle='--', alpha=0.5)
ax2.set_xlabel('Match Number', fontsize=12)
ax2.set_ylabel('ROI (%)', fontsize=12)
ax2.set_title('ROI Evolution', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observe how strategies perform over the full season!")

---

# 3. Understanding ROI

**Return on Investment (ROI)** is the ultimate metric for betting success.

## Formula

```
ROI = (Total Profit / Total Staked) × 100%
```

### What's a good ROI?

- **0%**: Break even
- **2-5%**: Excellent (professional level)
- **5-10%**: Outstanding (rare)
- **>10%**: Suspicious (likely unsustainable)

### Why?

Bookmaker margin is typically 3-7%, so beating it by even 2-3% is impressive!

In [ ]:
def calculate_roi_metrics(results_df):
    """
    Calculate comprehensive ROI metrics.
    """
    total_bets = len(results_df)
    wins = results_df['correct'].sum()
    losses = total_bets - wins
    
    total_staked = results_df['stake'].sum()
    total_profit = results_df['profit'].sum()
    total_returns = total_staked + total_profit
    
    roi = (total_profit / total_staked) * 100
    win_rate = (wins / total_bets) * 100
    
    avg_win = results_df[results_df['correct']]['profit'].mean() if wins > 0 else 0
    avg_loss = results_df[~results_df['correct']]['profit'].mean() if losses > 0 else 0
    
    return {
        'Total Bets': total_bets,
        'Wins': wins,
        'Losses': losses,
        'Win Rate (%)': win_rate,
        'Total Staked': total_staked,
        'Total Returns': total_returns,
        'Total Profit': total_profit,
        'ROI (%)': roi,
        'Avg Win': avg_win,
        'Avg Loss': avg_loss,
        'Profit Factor': abs(avg_win / avg_loss) if avg_loss != 0 else 0
    }

# Calculate for all strategies
metrics_home = calculate_roi_metrics(results_home)
metrics_lowest = calculate_roi_metrics(results_lowest)
if len(results_combined) > 0:
    metrics_combined = calculate_roi_metrics(results_combined)

print("Detailed ROI Metrics:\n")
print("Always Home Strategy:")
for key, value in metrics_home.items():
    if 'Rate' in key or 'ROI' in key:
        print(f"  {key}: {value:.2f}%")
    elif 'Avg' in key or 'Profit' in key or 'Factor' in key:
        print(f"  {key}: ${value:.2f}" if 'Factor' not in key else f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value:,.0f}")

print("\nLowest Odds Strategy:")
for key, value in metrics_lowest.items():
    if 'Rate' in key or 'ROI' in key:
        print(f"  {key}: {value:.2f}%")
    elif 'Avg' in key or 'Profit' in key or 'Factor' in key:
        print(f"  {key}: ${value:.2f}" if 'Factor' not in key else f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value:,.0f}")

## Summary

In this notebook, you learned:

✅ **Baseline strategies** - Always home, lowest odds, combined

✅ **Accuracy vs profitability** - High accuracy ≠ high profit

✅ **ROI calculation** - The ultimate profitability metric

✅ **Performance tracking** - Visualizing bankroll evolution

✅ **Comprehensive metrics** - Win rate, profit factor, etc.

### Key Takeaways

**1. Accuracy is NOT enough**
- 60% accuracy can still lose money
- Odds matter as much as predictions
- Bookmaker margin must be overcome

**2. ROI is king**
- Measures true profitability
- Accounts for stake size
- Industry standard metric

**3. Baselines are essential**
- Establish what to beat
- Simple strategies can be effective
- Complex models must add value

**4. Track performance**
- Monitor cumulative profit
- Watch for losing streaks
- Adjust strategies based on data

## Practice Exercises

1. **New baseline**: Create "always bet draw" strategy and calculate ROI

2. **Selective betting**: Only bet when home odds < 2.0. How does ROI change?

3. **Confidence threshold**: Only bet when favorite has odds < 1.8

4. **Streak analysis**: Find longest winning and losing streaks

5. **Monthly performance**: Calculate ROI by month (assuming 30-40 matches/month)

6. **Risk metrics**: Calculate maximum drawdown for each strategy

7. **Odds ranges**: Compare ROI for bets at different odds ranges (1.5-2.0, 2.0-3.0, etc.)

8. **Combination strategy**: Create your own strategy combining multiple factors